In [ ]:
import dynamiqs as dq
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# --- constants (computed once) ---
ħ = 1
e = 1
phi_quantum = 1/2

w_a = 3.8 * 2 * jnp.pi #GHz
w_b = 2*w_a#7.05 * 2 * jnp.pi #GHz

phi_a = 0.11
Z_a = 2 * phi_quantum**2 * phi_a**2 
C_a_inv = Z_a * w_a
E_Ca = C_a_inv / 2

phi_b = 0.2
Z_b = 2 * phi_quantum**2 * phi_b**2 
C_b_inv = Z_b * w_b
E_Cb = C_b_inv / 2

r=phi_b/phi_a

E_J = 37 * 2 * jnp.pi*1.5

In [ ]:
from scipy.special import jv
epsilon_0 = 0.5*np.pi
g2 = jv(1,epsilon_0)*E_J*phi_a**2*phi_b # (Leon's paper)

sin_pump = E_J*np.sin(epsilon_0)*phi_a**2*phi_b

#alpha = jnp.sqrt(epsilon_d/(2*g2))

In [ ]:
n_a = 60
n_b = n_a
a = dq.destroy(n_a)
b = dq.destroy(n_b)

varphi_a_op_tot = dq.tensor(jnp.sqrt(Z_a/2)*(a + a.dag()) / phi_quantum, dq.eye(n_b))
varphi_b_op_tot = dq.tensor(dq.eye(n_a), jnp.sqrt(Z_b/2)*(b + b.dag()) / phi_quantum)

psi_0 = dq.tensor(dq.fock(n_a, 1), dq.fock(n_b, 0))

In [ ]:
x_a = a+dq.dag(a)
p_a = dq.dag(a) - a
x_b = b+dq.dag(b)
p_b = dq.dag(b) - b
H_pot = 0.01*(dq.tensor(x_a.powm(4)+p_a.powm(4), dq.eye(n_b))+r*(dq.tensor(dq.eye(n_a), x_b.powm(4)+p_b.powm(4))))

In [ ]:
H_coupling = sin_pump*(dq.tensor(a@a, dq.dag(b))+r**2*dq.tensor(dq.eye(n_a), b@b@dq.dag(b)))

H = dq.tensor(w_a * dq.number(n_a), dq.eye(n_b)) \
    + dq.tensor(dq.eye(n_a), w_b * dq.number(n_b)) \
    + H_coupling + dq.dag(H_coupling) \
    + H_pot

In [ ]:
eigvals = np.real(np.linalg.eigvals(H))

In [ ]:
# Prepare array for results
z_list = []

for k, lam in enumerate(eigvals):

    # Distances from λ_k to all others
    distances = np.abs(eigvals - lam)

    # Exclude λ_k itself by setting its distance to +∞
    distances[k] = np.inf
    
    # Get indices of the two nearest neighbors (NN and NNN)
    nn_index = np.argmin(distances)
    nn_dist = distances[nn_index]

    # Temporarily ignore the NN to find the NNN
    distances[nn_index] = np.inf
    nnn_index = np.argmin(distances)

    # Compute the complex ratio z_k
    z = (eigvals[nn_index] - lam) / (eigvals[nnn_index] - lam)
    z_list.append(z)

z_list = np.array(z_list)

In [ ]:
len(z_list)

In [ ]:
plt.figure(figsize=(6,4))

plt.hist(z_list, bins=40, density=True, color='cornflowerblue', alpha=0.8)
plt.xlabel(r'$|z|$')
plt.ylabel('Probability density')
plt.title('Radial distribution of $z_k$')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,4))

plt.hist(z_list, bins=20, density=True, color='cornflowerblue', alpha=0.8)
plt.xlabel(r'$|z|$')
plt.ylabel('Probability density')
plt.title('Radial distribution of $z_k$')
plt.grid(True)
plt.tight_layout()
plt.show()

# Liouvillian

In [ ]:
n_a = 14
n_b = (n_a + 1) // 2
a = dq.destroy(n_a)
b = dq.destroy(n_b)

varphi_a_op_tot = dq.tensor(jnp.sqrt(Z_a/2)*(a + a.dag()) / phi_quantum, dq.eye(n_b))
varphi_b_op_tot = dq.tensor(dq.eye(n_a), jnp.sqrt(Z_b/2)*(b + b.dag()) / phi_quantum)

psi_0 = dq.tensor(dq.fock(n_a, 1), dq.fock(n_b, 0))

w_d = w_b
drive = lambda t : epsilon_d*jnp.cos(w_d*t)
H_drive = dq.modulated(drive, dq.tensor(dq.eye(n_a), b + b.dag()))

w_p=2*w_a-w_d
pump = lambda t : - 2 * E_J * jnp.sin(epsilon_0*jnp.cos(w_p*t))
H_pump = dq.modulated(pump, dq.sinm(varphi_a_op_tot + varphi_b_op_tot) - varphi_a_op_tot - varphi_b_op_tot)

H = dq.tensor(w_a * dq.number(n_a), dq.eye(n_b)) \
    + dq.tensor(dq.eye(n_a), w_b_0 * dq.number(n_b)) \
    + H_drive + H_pump

jumps = [jnp.sqrt(2*kappa) * dq.tensor(dq.eye(n_a), b)]

In [ ]:
t=1/kappa
L_H = -1j*np.array(dq.tensor(np.array(H(t)), dq.eye(n_a*n_b)) - np.array(dq.tensor(dq.eye(n_a*n_b), np.conj(H(t)))))
Diss = np.array(dq.tensor(jumps[0], np.conj(jumps[0]))) \
- 1/2 * np.array(dq.tensor(jumps[0].dag()@jumps[0], dq.eye(n_a*n_b))) \
- 1/2 * np.array(dq.tensor(dq.eye(n_a*n_b), np.transpose(jumps[0])@np.conj(jumps[0])))

L = L_H + Diss

In [ ]:
eigvals = np.linalg.eigvals(L)

In [ ]:
# Prepare array for results
z_list = []

for k, lam in enumerate(eigvals):

    # Distances from λ_k to all others
    distances = np.abs(eigvals - lam)

    # Exclude λ_k itself by setting its distance to +∞
    distances[k] = np.inf
    
    # Get indices of the two nearest neighbors (NN and NNN)
    nn_index = np.argmin(distances)
    nn_dist = distances[nn_index]

    # Temporarily ignore the NN to find the NNN
    distances[nn_index] = np.inf
    nnn_index = np.argmin(distances)

    # Compute the complex ratio z_k
    z = (eigvals[nn_index] - lam) / (eigvals[nnn_index] - lam)
    z_list.append(z)

z_list = np.array(z_list)

In [ ]:
import matplotlib.pyplot as plt
z = np.array(z_list)

x = z.real
y = z.imag

plt.figure(figsize=(6,5))

# 2D histogram
plt.hist2d(x, y, bins=50, density=True, cmap='viridis')

plt.colorbar(label='Density')

plt.xlabel(r'$\Re(z)$')
plt.ylabel(r'$\Im(z)$')
plt.title('Density of $z_k$ in the complex plane')

plt.tight_layout()
plt.show()

In [ ]:
r = np.abs(z)
plt.figure(figsize=(6,4))

plt.hist(r, bins=20, density=True, color='cornflowerblue', alpha=0.8)
plt.xlabel(r'$|z|$')
plt.ylabel('Probability density')
plt.title('Radial distribution of $z_k$')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
r = np.abs(z)
plt.figure(figsize=(6,4))

plt.hist(r, bins=20, density=True, color='cornflowerblue', alpha=0.8)
plt.xlabel(r'$|z|$')
plt.ylabel('Probability density')
plt.title('Radial distribution of $z_k$')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
g2

# Reproducing Michiel's results

In [ ]:
beta = 0.5
lambda_ = 0.3
nu_d = 2.35
xi_d = 1.85
drive_amp = xi_d /(np.sqrt(2)*lambda_)*(1-nu_d**2)/nu_d 

n_a = 50
a = dq.destroy(n_a)
x = (a.dag()+a)/np.sqrt(2)
p = 1j*(a.dag()-a)/np.sqrt(2)

drive = lambda t : drive_amp *jnp.cos(nu_d*t)
H_drive = dq.modulated(drive, p)

H = dq.number(n_a) - H_drive - beta/(2*lambda_**2)*dq.cosm(jnp.sqrt(2)*lambda_*x)

kappa = 0.5
jumps = [jnp.sqrt(2*kappa) * a]

In [ ]:
t=0/kappa
L_H = -1j*np.array(dq.tensor(np.array(H(t)), dq.eye(n_a)) - np.array(dq.tensor(dq.eye(n_a), np.conj(H(t)))))
Diss = np.array(dq.tensor(jumps[0], np.conj(jumps[0]))) \
- 1/2 * np.array(dq.tensor(jumps[0].dag()@jumps[0], dq.eye(n_a))) \
- 1/2 * np.array(dq.tensor(dq.eye(n_a), np.transpose(jumps[0])@np.conj(jumps[0])))

L = L_H + Diss
eigvals = np.linalg.eigvals(L)

In [ ]:
z_list = []

for k, lam in enumerate(eigvals):

    # Distances from λ_k to all others
    distances = np.abs(eigvals - lam)

    # Exclude λ_k itself by setting its distance to +∞
    distances[k] = np.inf
    
    # Get indices of the two nearest neighbors (NN and NNN)
    nn_index = np.argmin(distances)
    nn_dist = distances[nn_index]

    # Temporarily ignore the NN to find the NNN
    distances[nn_index] = np.inf
    nnn_index = np.argmin(distances)

    # Compute the complex ratio z_k
    z = (eigvals[nn_index] - lam) / (eigvals[nnn_index] - lam)
    z_list.append(z)

z_list = np.array(z_list)
r = np.abs(z_list)

In [ ]:
plt.figure(figsize=(6,4))

plt.hist(r, bins=20, density=True, color='cornflowerblue', alpha=0.8)
plt.xlabel(r'$|z|$')
plt.ylabel('Probability density')
plt.title('Radial distribution of $z_k$')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,4))

plt.hist(r, bins=20, density=True, color='cornflowerblue', alpha=0.8)
plt.xlabel(r'$|z|$')
plt.ylabel('Probability density')
plt.title('Radial distribution of $z_k$')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
beta = 0.5
lambda_ = 0.3
nu_d = 2.35
xi_d = 1.85
drive_amp = xi_d /(np.sqrt(2)*lambda_)*(1-nu_d**2)/nu_d 

n_a = 50
a = dq.destroy(n_a)
x = (a.dag()+a)/np.sqrt(2)
p = 1j*(a.dag()-a)/np.sqrt(2)

drive = lambda t : drive_amp *jnp.cos(nu_d*t)
H_drive = dq.modulated(drive, p)

H = dq.number(n_a) - H_drive - beta/(2*lambda_**2)*dq.cosm(jnp.sqrt(2)*lambda_*x)